> We are gonna create our first CSV file with my owned game that's gonna be used next step to retrieve album with the OSTs

In [ ]:
import os
import pandas as pd
from dotenv import load_dotenv
from steam_web_api import Steam

load_dotenv()

STEAM_KEY = os.getenv("STEAM_API_KEY")
steam = Steam(STEAM_KEY)

# Prefer Steam ID from environment, otherwise prompt the user
MY_STEAM_ID = os.getenv("MY_STEAM_ID")
if not MY_STEAM_ID:
    MY_STEAM_ID = input("Enter your Steam ID (or press Enter to use a redacted example dataset): ").strip()

# I removed my personal SteamID from the notebook to protect privacy.
# If you don't provide your own Steam ID (via env or input), a small redacted example dataset
# will be created so you can run the pipeline and see how it works locally.
if not MY_STEAM_ID:
    # Write a tiny redacted example so downstream cells can run without real profile data
    games_df = pd.DataFrame([
        {"appid": 123456, "name": "Example Game (redacted)", "playtime_forever": 0}
    ])
    os.makedirs("../data", exist_ok=True)
    games_df.to_csv("../data/steam_games.csv", index=False)
    print("No Steam ID provided — an example redacted dataset was written to ../data/steam_games.csv")
else:
    owned_games = steam.users.get_owned_games(MY_STEAM_ID)
    games_list = owned_games.get("games", [])
    games_df = pd.DataFrame(games_list)
    games_df = games_df[["appid", "name", "playtime_forever"]].copy()
    os.makedirs("../data", exist_ok=True)
    games_df.to_csv("../data/steam_games.csv", index=False)
    print(f"Saved {len(games_df)} games to data/steam_games.csv")
    print(games_df.head(10))

In [ ]:
import os
import time
import pandas as pd
import requests
from dotenv import load_dotenv
import spotipy
from spotipy.oauth2 import SpotifyClientCredentials

load_dotenv()

RAPIDAPI_KEY = os.getenv("RAPIDAPI_KEY")
RAPIDAPI_HOST = "track-analysis.p.rapidapi.com"
CLIENT_ID = os.getenv("SPOTIFY_CLIENT_ID")
CLIENT_SECRET = os.getenv("SPOTIFY_CLIENT_SECRET")
SAMPLE_SIZE = 35
MAX_RETRIES = 3
RETRY_BASE_SLEEP = 0.3

if not RAPIDAPI_KEY or not CLIENT_ID or not CLIENT_SECRET:
    raise SystemExit("Missing RAPIDAPI_KEY or Spotify credentials in .env")

sp = spotipy.Spotify(
    auth_manager=SpotifyClientCredentials(client_id=CLIENT_ID, client_secret=CLIENT_SECRET),
    requests_timeout=20,
 )

session = requests.Session()

cache_path = os.path.normpath("../data/rapidapi_cache.csv")
if os.path.exists(cache_path):
    cache_df = pd.read_csv(cache_path)
    cached_ids = set(cache_df["spotify_track_id"].astype(str).str.strip().tolist())
else:
    cache_df = pd.DataFrame()
    cached_ids = set()

games_df = pd.read_csv(os.path.normpath("../data/steam_games.csv"))
games_df = games_df.sort_values("playtime_forever", ascending=False).reset_index(drop=True)
sample_size = min(SAMPLE_SIZE, len(games_df))
selected = games_df.head(sample_size)

rows = []
required_fields = ["danceability", "energy", "tempo"]

for _, g in selected.iterrows():
    title = g["name"]
    q = f"{title} soundtrack"  # helps prioritize OST albums
    res = sp.search(q=q, type="album", limit=3)
    items = res.get("albums", {}).get("items", [])
    if not items:
        continue
    album = items[0]
    album_name = album.get("name")
    album_id = album.get("id")
    tracks = sp.album_tracks(album_id, limit=50).get("items", [])
    for t in tracks:  # if needed, limit with tracks[:10]
        track_name = t.get("name")
        print(f"--->    Currently fetching track {track_name}")
        track_id = t.get("id")
        if not track_id or str(track_id).strip() in cached_ids:
            continue

        url = f"https://{RAPIDAPI_HOST}/pktx/spotify/{track_id}"
        headers = {"x-rapidapi-key": RAPIDAPI_KEY, "x-rapidapi-host": RAPIDAPI_HOST}

        j = None
        for attempt in range(1, MAX_RETRIES + 1): # retry if issues with the transmission
            try:
                r = session.get(url, headers=headers, timeout=30)
                r.raise_for_status()
                j = r.json()
                if any(j.get(k) is None for k in required_fields):
                    if attempt < MAX_RETRIES:
                        wait_s = RETRY_BASE_SLEEP * attempt
                        print(f"--->    Incomplete payload for {track_name}, retry {attempt}/{MAX_RETRIES} after {wait_s:.1f}s")
                        time.sleep(wait_s)
                        continue
                    j = None
                break
            except requests.RequestException as e:
                if attempt < MAX_RETRIES:
                    wait_s = RETRY_BASE_SLEEP * attempt
                    print(f"--->    Request failed for {track_name}, retry {attempt}/{MAX_RETRIES} after {wait_s:.1f}s ({e})")
                    time.sleep(wait_s)
                else:
                    print(f"--->    Exception for track {track_name}: giving up after {MAX_RETRIES} tries")

        if j is None:
            print(f"--->    Track {track_name}: not cached (failed/incomplete)")
            continue

        row = {
            "game_title": title,
            "appid": g["appid"],
            "album_name": album_name,
            "spotify_track_id": track_id,
            "track_name": track_name,
            "danceability": j.get("danceability"),
            "energy": j.get("energy"),
            "valence": j.get("happiness") or j.get("valence"),
            "acousticness": j.get("acousticness"),
            "instrumentalness": j.get("instrumentalness"),
            "liveness": j.get("liveness"),
            "speechiness": j.get("speechiness"),
            "tempo": j.get("tempo"),
        }
        rows.append(row)
        df_row = pd.DataFrame([row])
        header = not os.path.exists(cache_path)
        df_row.to_csv(cache_path, mode="a", index=False, header=header)
        cached_ids.add(str(track_id).strip())
        print(f"--->    Track {track_name}: FETCHED !")
        time.sleep(0.4)

if rows:
    out_df = pd.DataFrame(rows)
    out_path = os.path.normpath("../data/rapidapi_features.csv")

    if os.path.exists(cache_path):
        cache_df = pd.read_csv(cache_path)
        num_cols = ["danceability", "energy", "valence", "acousticness", "instrumentalness", "liveness", "speechiness", "tempo"]
        for col in num_cols:
            cache_df[col] = pd.to_numeric(cache_df[col], errors="coerce")

        valid = cache_df.dropna(subset=["danceability", "energy", "tempo"]).copy()

        if "spotify_track_id" in valid.columns:
            valid = valid.drop_duplicates(subset=["spotify_track_id"]).reset_index(drop=True)

        valid.to_csv(out_path, index=False)
        print(f"Rebuilt {out_path} from cache: {len(valid)} valid feature rows")
    else:
        out_df.to_csv(out_path, index=False)
        print(f"Saved {len(out_df)} feature rows to {out_path}")
else:
    print("No new feature rows collected")

> If you want to include games that aren't on your steam this next section is for you !

> We're gonna make a list of names of games we want to include in our dataset and then do request to RapidAPI again !

In [ ]:
import os
import time
import pandas as pd
import requests
from dotenv import load_dotenv
import spotipy
from spotipy.oauth2 import SpotifyClientCredentials

load_dotenv()

# You can put a list of whatever additional games that you want to add on the dataset !
additional_games = ["Genshin Impact", "Splatoon", "Animal Crossing: New Horizons", "Overwatch"]

RAPIDAPI_KEY = os.getenv("RAPIDAPI_KEY")
RAPIDAPI_HOST = os.getenv("RAPIDAPI_HOST", "track-analysis.p.rapidapi.com")
CLIENT_ID = os.getenv("SPOTIFY_CLIENT_ID")
CLIENT_SECRET = os.getenv("SPOTIFY_CLIENT_SECRET")
SAMPLE_SIZE = 35
MAX_RETRIES = 3
RETRY_BASE_SLEEP = 0.3
required_fields = ["danceability", "energy", "tempo"]

if not RAPIDAPI_KEY or not CLIENT_ID or not CLIENT_SECRET:
    raise SystemExit("Missing RAPIDAPI_KEY or Spotify credentials in .env")

session = requests.Session()
sp = spotipy.Spotify(auth_manager=SpotifyClientCredentials(client_id=CLIENT_ID, client_secret=CLIENT_SECRET), requests_timeout=20)

cache_path = os.path.normpath("../data/rapidapi_cache.csv")
if os.path.exists(cache_path):
    cache_df = pd.read_csv(cache_path)
    cached_ids = set(cache_df["spotify_track_id"].astype(str).str.strip().tolist())
else:
    cached_ids = set()

rows = []

for game in additional_games:
    query = f"{game} soundtrack"
    res = sp.search(q=query, type="album", limit=3)
    items = res.get("albums", {}).get("items", [])

    if not items:
        continue

    album = items[0]
    album_name = album.get("name")
    album_id = album.get("id")
    tracks = sp.album_tracks(album_id, limit=50).get("items", [])

    for t in tracks:
        track_name = t.get("name")
        track_id = t.get("id")
        print(f"--->    Currently fetching track {track_name}")

        if not track_id or str(track_id).strip() in cached_ids:
            continue

        url = f"https://{RAPIDAPI_HOST}/pktx/spotify/{track_id}"
        headers = {"x-rapidapi-key": RAPIDAPI_KEY, "x-rapidapi-host": RAPIDAPI_HOST}
        j = None

        for attempt in range(1, MAX_RETRIES + 1):  # retry if issues with the transmission
            try:
                r = session.get(url, headers=headers, timeout=30)
                r.raise_for_status()
                j = r.json()
                if any(j.get(k) is None for k in required_fields):
                    if attempt < MAX_RETRIES:
                        wait_s = RETRY_BASE_SLEEP * attempt
                        print(f"--->    Incomplete payload for {track_name}, retry {attempt}/{MAX_RETRIES} after {wait_s:.1f}s")
                        time.sleep(wait_s)
                        continue
                    j = None
                break
            except requests.RequestException as e:
                if attempt < MAX_RETRIES:
                    wait_s = RETRY_BASE_SLEEP * attempt
                    print(f"--->    Request failed for {track_name}, retry {attempt}/{MAX_RETRIES} after {wait_s:.1f}s ({e})")
                    time.sleep(wait_s)
                else:
                    print(f"--->    Exception for track {track_name}: giving up after {MAX_RETRIES} tries")

        if j is None:
            print(f"--->    Track {track_name}: not cached (failed/incomplete)")
            continue

        row = {
            "game_title": game,
            "appid": pd.NA,
            "album_name": album_name,
            "spotify_track_id": track_id,
            "track_name": track_name,
            "danceability": j.get("danceability"),
            "energy": j.get("energy"),
            "valence": j.get("happiness") or j.get("valence"),
            "acousticness": j.get("acousticness"),
            "instrumentalness": j.get("instrumentalness"),
            "liveness": j.get("liveness"),
            "speechiness": j.get("speechiness"),
            "tempo": j.get("tempo"),
        }
        rows.append(row)
        df_row = pd.DataFrame([row])
        header = not os.path.exists(cache_path)
        df_row.to_csv(cache_path, mode="a", index=False, header=header)
        cached_ids.add(str(track_id).strip())
        print(f"--->    Track {track_name}: FETCHED !")
        time.sleep(0.4)

if rows:
    out_df = pd.DataFrame(rows)
    out_path = os.path.normpath("../data/rapidapi_features.csv")
    if os.path.exists(cache_path):
        cache_df = pd.read_csv(cache_path)
        num_cols = ["danceability", "energy", "valence", "acousticness", "instrumentalness", "liveness", "speechiness", "tempo"]
        for col in num_cols:
            cache_df[col] = pd.to_numeric(cache_df[col], errors="coerce")
        valid = cache_df.dropna(subset=["danceability", "energy", "tempo"]).copy()
        if "spotify_track_id" in valid.columns:
            valid = valid.drop_duplicates(subset=["spotify_track_id"]).reset_index(drop=True)
        valid.to_csv(out_path, index=False)
        print(f"Rebuilt {out_path} from cache: {len(valid)} valid feature rows")
    else:
        out_df.to_csv(out_path, index=False)
        print(f"Saved {len(out_df)} feature rows to {out_path}")
else:
    print("No new feature rows collected")